# Image Map Annotation

Label an image collection by sampling across a 2D map of the images, so that partial labels cover every kind of image in the set.

Each image's feature vector is projected to 2D with UMAP. A grid is laid over the map and you label one image per grid cell. A linear classifier trained on your labels predicts every unlabeled image, and from the second round on each grid cell offers the image it is least sure about. After each round the map is recomputed using your labels only (semi-supervised UMAP).

Input: a folder of images and a DataFrame `images_df` with an `image` column (file name) and a `features` column (one vector per image), plus any metadata you want to keep. The example section builds both from 1,500 MNIST digits.

Output: `outputs/labels.csv` with your labels, saved on every click, and `outputs/predictions.csv` with a predicted label and confidence for every image.

## Setup

In [ ]:
# %pip install -q numpy pandas pillow scikit-learn umap-learn ipywidgets "jupyter-scatter[annotation-extras]" plotly nbformat

In [ ]:
import base64
import gzip
import io
import warnings
from pathlib import Path

import ipywidgets as widgets
import jscatter
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from umap import UMAP

try:  # Colab only
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

warnings.filterwarnings("ignore", category=UserWarning, module="umap")
warnings.filterwarnings("ignore", category=FutureWarning, module="umap")

## Functions

In [ ]:
def download_mnist_example(out_dir="example_data", per_class=150, seed=0):
    import urllib.request

    out_dir = Path(out_dir)
    raw_dir, image_dir = out_dir / "mnist_raw", out_dir / "images"
    raw_dir.mkdir(parents=True, exist_ok=True)
    image_dir.mkdir(parents=True, exist_ok=True)

    def read_idx(name):
        path = raw_dir / name
        if not path.exists():
            urllib.request.urlretrieve(
                "https://storage.googleapis.com/cvdf-datasets/mnist/" + name, path
            )
        data = gzip.open(path).read()
        ndim = data[3]
        shape = [
            int.from_bytes(data[4 + 4 * i : 8 + 4 * i], "big") for i in range(ndim)
        ]
        return np.frombuffer(data, dtype=np.uint8, offset=4 + 4 * ndim).reshape(shape)

    images = read_idx("train-images-idx3-ubyte.gz")
    digits = read_idx("train-labels-idx1-ubyte.gz")

    rng = np.random.default_rng(seed)
    rows = []
    for digit in range(10):
        for i in rng.choice(
            np.flatnonzero(digits == digit), size=per_class, replace=False
        ):
            Image.fromarray(images[i]).save(image_dir / f"{i:05d}.png")
            rows.append({"image": f"{i:05d}.png", "true_class": str(digit)})
    print(f"Saved {len(rows)} images to {image_dir}")
    return pd.DataFrame(rows).sort_values("image", ignore_index=True)


def pixel_features(path, size=16):
    img = Image.open(path).convert("L").resize((size, size))
    return np.asarray(img, dtype=np.float32).ravel() / 255


def umap_2d(
    features, n_neighbors=15, labels=None, label_weight=0.5, seed=42, pca_dims=50
):
    y = None
    if labels is not None:
        codes, _ = pd.factorize(pd.Series(labels, dtype="object"))  # unlabeled -> -1
        y = codes if (codes >= 0).any() else None

    if pca_dims is not None and features.shape[1] > pca_dims:
        features = PCA(n_components=pca_dims, random_state=seed).fit_transform(features)

    features_umap = UMAP(
        n_neighbors=n_neighbors,
        min_dist=0.1,
        target_weight=label_weight,
        random_state=seed,
    ).fit_transform(features, y=y)
    return features_umap


def thumbnail(path, size=160):
    img = Image.open(path).convert("RGB")
    img.thumbnail((size, size))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def map_scatter(
    map_df, color_by=None, size_by=None, opacity_by=None, contours=False, height=500
):
    df = map_df.copy()
    if size_by is not None:
        df[size_by] = df[size_by].astype("category")
    s = jscatter.Scatter(data=df, x="x", y="y", height=height)
    s.axes(False)
    s.size(5)
    if color_by is not None and df[color_by].notna().any():
        s.color(by=color_by)
        s.legend(True)
        if contours:
            counts = df[color_by].value_counts()
            if counts.min() >= 5:
                s.annotations([jscatter.Contour(by=color_by)])
            else:
                print("Contours appear once every label has at least 5 images.")
    if size_by is not None:
        s.size(by=size_by, map=[4, 14])
    if opacity_by is not None:
        s.opacity(by=opacity_by, map=(0.1, 1.0, 10), norm=(0, 1))
    s.tooltip(True, preview="thumb", preview_type="image", preview_image_height=200)
    return s


def grid_sample(map_df, n, exclude=(), score=None):
    """One image from every non-empty cell of an n x n grid over the map, skipping `exclude`.
    Without `score`: the image nearest the cell centre. With `score` (one value per row of map_df):
    the image with the lowest score in the cell, e.g. the lowest prediction confidence."""
    x, y = map_df["x"].to_numpy(), map_df["y"].to_numpy()
    x_edges = np.linspace(x.min(), x.max(), n + 1)
    y_edges = np.linspace(y.min(), y.max(), n + 1)
    cell_x = np.clip(np.digitize(x, x_edges[1:-1]), 0, n - 1)
    cell_y = np.clip(np.digitize(y, y_edges[1:-1]), 0, n - 1)
    available = ~map_df["image"].isin(set(exclude)).to_numpy()

    picked = []
    for gx in range(n):
        for gy in range(n):
            idx = np.flatnonzero((cell_x == gx) & (cell_y == gy) & available)
            if not len(idx):
                continue
            if score is None:
                cx = (x_edges[gx] + x_edges[gx + 1]) / 2
                cy = (y_edges[gy] + y_edges[gy + 1]) / 2
                picked.append(idx[np.argmin((x[idx] - cx) ** 2 + (y[idx] - cy) ** 2)])
            else:
                picked.append(idx[np.argmin(score[idx])])

    sample = map_df.iloc[picked].copy()
    sample["cell_x"], sample["cell_y"] = cell_x[picked], cell_y[picked]
    return sample.reset_index(drop=True)


def image_grid(map_df, sample_df, n, size=800):
    x, y = map_df["x"].to_numpy(), map_df["y"].to_numpy()
    x_edges = np.linspace(x.min(), x.max(), n + 1)
    y_edges = np.linspace(y.min(), y.max(), n + 1)
    fig = go.Figure(
        go.Scattergl(
            x=x,
            y=y,
            mode="markers",
            hoverinfo="none",
            marker=dict(size=2, color="lightgray", opacity=0.4),
        )
    )
    for row in sample_df.itertuples():
        x0, x1 = x_edges[row.cell_x], x_edges[row.cell_x + 1]
        y0, y1 = y_edges[row.cell_y], y_edges[row.cell_y + 1]
        fig.add_layout_image(
            source=row.thumb,
            xref="x",
            yref="y",
            x=x0,
            y=y1,
            sizex=x1 - x0,
            sizey=y1 - y0,
            sizing="stretch",
            layer="above",
        )
        fig.add_shape(
            type="rect", x0=x0, y0=y0, x1=x1, y1=y1, line=dict(color="black", width=1)
        )
    fig.update_layout(
        title=f"One image per cell ({n}x{n} grid)",
        width=size,
        height=size,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        template="plotly_white",
        showlegend=False,
    )
    return fig


def load_labels(path):
    if not Path(path).exists():
        return {}
    df = pd.read_csv(path, dtype=str)
    return dict(zip(df["image"], df["label"]))


def save_labels(labels, path):
    pd.DataFrame(labels.items(), columns=["image", "label"]).to_csv(path, index=False)


def fit(img, size):
    scale = size / max(img.size)
    new_size = (max(1, round(img.width * scale)), max(1, round(img.height * scale)))
    return img.resize(new_size, Image.NEAREST if scale > 1 else Image.LANCZOS)


def annotate(batch_df, categories, labels, save_path, image_height=300):
    items = batch_df[["image", "path"]].to_dict("records")
    state = {"i": 0}
    progress, caption = widgets.HTML(), widgets.HTML()
    picture = widgets.Image(format="jpeg")

    def show():
        i = state["i"]
        done = sum(item["image"] in labels for item in items)
        progress.value = f"<b>{done} / {len(items)} labeled in this batch</b>"
        if i >= len(items):
            picture.layout.display = "none"
            caption.value = "Batch finished. Run the next cell."
            return
        picture.layout.display = None
        img = Image.open(items[i]["path"]).convert("RGB")
        img = fit(
            img, round(image_height * max(img.size) / img.height)
        )  # keep aspect ratio
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=90)
        picture.value = buf.getvalue()
        picture.width, picture.height = str(img.width), str(img.height)
        current = labels.get(items[i]["image"])
        caption.value = (
            f"{i + 1}/{len(items)} &nbsp; <code>{items[i]['image']}</code>"
            + (f" &nbsp; current label: <b>{current}</b>" if current else "")
        )

    def on_label(button):
        if state["i"] < len(items):
            labels[items[state["i"]]["image"]] = button.description
            save_labels(labels, save_path)
            state["i"] += 1
            show()

    def move(step):
        state["i"] = min(max(state["i"] + step, 0), len(items))
        show()

    buttons = [widgets.Button(description=c, button_style="info") for c in categories]
    for b in buttons:
        b.on_click(on_label)
    skip, back = widgets.Button(description="Skip"), widgets.Button(description="Back")
    skip.on_click(lambda _: move(1))
    back.on_click(lambda _: move(-1))
    show()
    return widgets.VBox(
        [
            progress,
            widgets.HBox(
                buttons + [skip, back], layout=widgets.Layout(flex_flow="row wrap")
            ),
            picture,
            caption,
        ]
    )


def predict_labels(features, labels):
    """Logistic regression on the (standardised) features, trained on the images labelled so far.
    `confidence` is the probability of the predicted class (1.0 for images you labelled)."""
    labels = pd.Series(labels, dtype="object").reset_index(drop=True)
    known = labels.notna().to_numpy()
    if labels[known].nunique() < 2:
        print("Label images from at least two classes in step 2b, then run this again.")
        return pd.DataFrame({"predicted": [None] * len(labels), "confidence": 0.0})
    x = StandardScaler().fit(features[known]).transform(features)
    clf = LogisticRegression(max_iter=3000, C=0.5).fit(x[known], labels[known])
    proba = clf.predict_proba(x)
    predicted = clf.classes_[proba.argmax(axis=1)].astype(object)
    confidence = proba.max(axis=1)
    predicted[known], confidence[known] = labels[known].to_numpy(), 1.0
    return pd.DataFrame({"predicted": predicted, "confidence": confidence})

## Settings

In [ ]:
# PATH_TO_YOUR_DATA = "..."  # path to your CSV file
IMAGE_DIR = "example_data/images"  # folder containing the files in images_df["image"]
COLOR_BY = "true_class"  # images_df column to colour the first map by, or None

CATEGORIES = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]


OUTPUT_DIR = "outputs"
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
LABELS_CSV = output_dir / "labels.csv"

## Your data

`images_df` needs one row per image with:

- `image`: file name inside `IMAGE_DIR`
- `features`: a 1D array, the same length for every image

Any other columns are kept and can be used for `COLOR_BY`. If you already have embeddings saved, this can be a single line such as `images_df = pd.read_parquet("embeddings.parquet")`.

In [ ]:
# images_df = pd.read_csv(PATH_TO_YOUR_DATA, index_col=0)

## Example: MNIST

### Download MNIST

Saves 150 images per digit from the CVDF mirror (`storage.googleapis.com/cvdf-datasets/mnist/`) (about 10 MB) to `example_data/images/`.

In [ ]:
example_df = download_mnist_example()
example_df.head()

### Get pixel features

Each image is resized to 16×16 grayscale and flattened to 256 values.

In [ ]:
images_df = example_df.assign(
    features=[pixel_features(Path(IMAGE_DIR) / name) for name in example_df["image"]],
    true_class=example_df["true_class"].astype("category"),
)
images_df.head()

## Check the input

In [ ]:
assert {"image", "features"} <= set(images_df.columns), (
    "images_df needs columns 'image' and 'features'"
)
images_df = images_df.reset_index(drop=True)
images_df["path"] = [str(Path(IMAGE_DIR) / name) for name in images_df["image"]]

missing = [p for p in images_df["path"] if not Path(p).is_file()]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} of {len(images_df)} images not found in {IMAGE_DIR}, e.g. {missing[:3]}"
    )

features = np.stack(images_df["features"].to_numpy()).astype(np.float32)
print(f"{len(images_df)} images, {features.shape[1]} features each")


## 1. Map

Hover over a point to see its image. Nearby points should look alike; if they don't, change `N_NEIGHBORS` or use better features.

In [ ]:
xy = umap_2d(features, n_neighbors=15)
map_df = images_df.drop(columns="features").assign(x=xy[:, 0], y=xy[:, 1])
map_df["thumb"] = [thumbnail(p) for p in map_df["path"]]

map_scatter(map_df, color_by=COLOR_BY, contours=True)

### 1b. One image per grid cell (optional)

The images that round 1 will ask you to label, drawn in place on the map.

In [ ]:
GRID_N = 8
image_grid(map_df, grid_sample(map_df, GRID_N), GRID_N).show()

## 2. Label in rounds

Run 2a once. It reloads labels saved earlier, so you can stop and resume. Then repeat 2b to 2d:

- 2b: label one unlabeled image per grid cell. In the first round it is the image nearest each cell's centre; after that, the image the classifier is least confident about. Each click is written to `labels.csv`.
- 2c: recompute the map with your labels and show progress. Only your labels shape the map; predictions are not used.
- 2d: a linear classifier (logistic regression on the features) trained on your labels predicts the remaining images. Faint points are low confidence, which is where the next round helps most.

Stop when the predictions look right.

In [ ]:
# 2a. Load labels saved earlier (if any)
labels = load_labels(LABELS_CSV)
print(f"{len(labels)} images already labeled")

In [ ]:
# 2b. One unlabeled image per grid cell; click a label for each.
#     First round: the image nearest each cell's centre. Later rounds: the image the
#     classifier from 2d is least confident about.
GRID_N = 8
score = map_df["confidence"].to_numpy() if labels and "confidence" in map_df else None
batch_df = grid_sample(map_df, GRID_N, exclude=labels, score=score)
print(
    f"This round: {len(batch_df)} images  |  unlabeled left: {len(map_df) - len(labels)}"
)

annotate(batch_df, CATEGORIES, labels, save_path=LABELS_CSV)

In [ ]:
# 2c. Recompute the map with your labels. Labeled images are drawn larger.
map_df["label"] = map_df["image"].map(labels).astype("category")
map_df["labeled"] = map_df["label"].notna()

xy = umap_2d(features, n_neighbors=15, labels=map_df["label"], label_weight=0.5)
map_df["x"], map_df["y"] = xy[:, 0], xy[:, 1]

display(map_df["label"].value_counts().to_frame("count"))
map_scatter(map_df, color_by="label", size_by="labeled", contours=True, height=600)

In [ ]:
# 2d. Predict each unlabeled image with a linear classifier trained on your labels (on the features).
pred = predict_labels(features, map_df["label"])
map_df["predicted"] = pred["predicted"].astype("category").to_numpy()
map_df["confidence"] = pred["confidence"].to_numpy()

map_df[["image", "label", "predicted", "confidence"]].to_csv(
    output_dir / "predictions.csv", index=False
)
unlabeled = ~map_df["labeled"]
print(f"Saved predictions -> {output_dir / 'predictions.csv'}")
print(
    f"Low confidence (< 0.9): {(map_df.loc[unlabeled, 'confidence'] < 0.9).mean():.0%} of unlabeled images"
)
map_scatter(map_df, color_by="predicted", opacity_by="confidence", height=600)